# Munsiq — Fine-Tuning Qwen2.5-7B for Arabic Contract JSON Extraction

Fine-tunes `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` with LoRA (Unsloth) to extract a
fixed JSON schema from bilingual (Saudi Arabic / English) industrial procurement
documents — Purchase Orders, Invoices, LPOs, Work Orders, Maintenance Logs,
Delivery Notes, RFQs, Warranty Claims, Service Reports, and Inspection Reports.

**Pipeline:** mount data → inspect dataset → configure → load model → attach LoRA →
build prompts → split train/eval → train (3 epochs, eval every N steps, no
checkpoint I/O) → save adapter → run inference → plot train vs. eval loss.


## 1. Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes


## 2. Configuration

Every path, hyperparameter, and constant lives here — nothing is hard-coded again
below. Change the data path if your Drive layout differs.


In [ ]:
from dataclasses import dataclass

@dataclass
class CFG:
    # Data
    data_path: str = "/content/drive/MyDrive/munsiq/data/procurement_finetune.jsonl"
    eval_split: float = 0.10
    seed: int = 3407

    # Model
    model_name: str = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
    max_seq_length: int = 2048
    load_in_4bit: bool = True

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1
    eval_steps: int = 20
    logging_steps: int = 1
    output_dir: str = "outputs"

cfg = CFG()


## 3. Data understanding (EDA)

Before writing a single prompt template, look at what's actually in the dataset:
how many examples, which `document_type` values exist, what the JSON schema
looks like field-by-field, and whether any field needs special handling
(e.g. inconsistent date formats) before it's baked into the prompt.


In [ ]:
import json
from collections import Counter

records = []
with open(cfg.data_path.replace("/content/drive/MyDrive", "/content/drive/MyDrive"), encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Total examples: {len(records)}")

doc_types = Counter(r["extracted_json"]["document_type"] for r in records)
print("\ndocument_type distribution:")
for k, v in doc_types.most_common():
    print(f"  {k:<20} {v}")

text_lens = [len(r["text"]) for r in records]
print(f"\n`text` length (chars): min={min(text_lens)} max={max(text_lens)} avg={sum(text_lens)/len(text_lens):.0f}")

print("\nSample record:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

# Check date format consistency -- this matters for how we describe the
# critical_dates field in the prompt schema below.
date_formats = Counter()
for r in records:
    for d in r["extracted_json"]["critical_dates"]:
        date_formats["YYYY-MM-DD"] += 1 if len(d.split("-")) == 3 else 0
        date_formats["DD/MM/YYYY"] += 1 if "/" in d else 0
print(f"\ncritical_dates formats seen: {dict(date_formats)}")


**Findings used below:**

- 500 examples, perfectly balanced across 10 `document_type` values (50 each).
- Every record has exactly 5 target fields: `document_type` (str, one of 10
  categories), `company_name` (str, Arabic or English), `equipment_mentioned`
  (list[str]), `total_value_sar` (float), `critical_dates` (list[str]).
- Dates are **not** normalized in the source data — both `YYYY-MM-DD` and
  `DD/MM/YYYY` appear. The prompt schema below tells the model to preserve the
  date exactly as written rather than reformat it, matching the ground truth.
- `text` is short (a few hundred characters), so `max_seq_length = 2048` has
  comfortable headroom.


## 4. Load base model (4-bit) and attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.model_name,
    max_seq_length = cfg.max_seq_length,
    dtype = None,
    load_in_4bit = cfg.load_in_4bit,
)

print(f"Loaded {cfg.model_name}")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_r,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = cfg.lora_alpha,
    lora_dropout = cfg.lora_dropout,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = cfg.seed,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapters attached.")


## 5. Prompt template and JSON schema

The schema mirrors the dataset exactly (field names, types, and the 10 valid
`document_type` values found during EDA), so the model is never asked to
predict a field or category that doesn't exist in the training data.


In [ ]:
EOS_TOKEN = tokenizer.eos_token

JSON_SCHEMA_DESCRIPTION = """{
  "document_type": string, // one of: Purchase Order, Invoice, LPO, Work Order,
                            // Maintenance Log, Delivery Note, RFQ, Warranty Claim,
                            // Service Report, Inspection Report
  "company_name": string,
  "equipment_mentioned": string[],
  "total_value_sar": number,
  "critical_dates": string[]  // copy each date exactly as it appears in the text,
                              // do not reformat it
}"""

PROMPT_TEMPLATE = """أنت مهندس بيانات خبير متخصص في المستندات الصناعية والعقود. استخرج الحقول التالية من النص أدناه وأعدها بصيغة JSON صالحة فقط، مطابقة تماماً للمخطط المحدد، دون أي شرح إضافي.

### المخطط المطلوب (JSON Schema):
{schema}

### النص الأصلي:
{text}

### المخرجات (JSON):
{output}"""

print("Prompt template and schema defined.")


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for input_text, output_json in zip(examples["text"], examples["extracted_json"]):
        formatted = PROMPT_TEMPLATE.format(
            schema=JSON_SCHEMA_DESCRIPTION,
            text=input_text,
            output=json.dumps(output_json, ensure_ascii=False),
        ) + EOS_TOKEN
        texts.append(formatted)
    return {"text": texts}

print("Formatting function ready.")


## 6. Load dataset, split train/eval, and map

A 10% held-out split is created **before** training so we can track validation
loss and catch overfitting. `remove_columns` drops the original `text` /
`extracted_json` columns during `.map()` so the mapped dataset only carries the
final formatted `text` field instead of accumulating both versions in memory.


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files=cfg.data_path, split="train")
split_dataset = raw_dataset.train_test_split(test_size=cfg.eval_split, seed=cfg.seed)

mapped_train = split_dataset["train"].map(
    formatting_prompts_func,
    batched=True,
    remove_columns=raw_dataset.column_names,
)
mapped_eval = split_dataset["test"].map(
    formatting_prompts_func,
    batched=True,
    remove_columns=raw_dataset.column_names,
)

print(f"Train examples: {len(mapped_train)} | Eval examples: {len(mapped_eval)}")


## 7. Trainer configuration

Key fixes vs. the original notebook:

- **No `max_steps`** — `num_train_epochs = 3` so all 500 examples are seen
  a fixed number of times regardless of batch size.
- **`save_strategy = "no"`** — the original `PicklingError` came from trying to
  checkpoint mid-training; since we only need the final adapter (saved
  manually in the next cell), no intermediate checkpoints are written.
- **`eval_dataset` + `eval_strategy = "steps"`** — validation loss is measured
  throughout training, not just at the end, giving enough points to see an
  overfitting trend in the plot below.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = mapped_train,
    eval_dataset = mapped_eval,
    dataset_text_field = "text",
    max_seq_length = cfg.max_seq_length,
    dataset_num_proc = 2,
    packing = False,

    args = TrainingArguments(
        per_device_train_batch_size = cfg.per_device_train_batch_size,
        per_device_eval_batch_size = cfg.per_device_eval_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        num_train_epochs = cfg.num_train_epochs,
        warmup_ratio = cfg.warmup_ratio,
        learning_rate = cfg.learning_rate,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = cfg.logging_steps,
        eval_strategy = "steps",
        eval_steps = cfg.eval_steps,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = cfg.seed,
        output_dir = cfg.output_dir,
        report_to = "none",
    ),
)

print("Trainer configured: 3 epochs, eval every "
      f"{cfg.eval_steps} steps, no intermediate checkpoints.")


In [ ]:
trainer_stats = trainer.train()


## 8. Save the LoRA adapter

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapter saved to ./lora_model")


## 9. Quick inference sanity check

In [ ]:
FastLanguageModel.for_inference(model)

new_contract_text = """
تم إبرام هذا العقد في يوم الخميس الموافق 20 مايو 2026، بين شركة أبعاد التقنية (الطرف الأول)،
ومؤسسة الإعمار الحديثة (الطرف الثاني). يتفق الطرفان على توريد معدات حاسوبية بقيمة إجمالية قدرها
150,000 ريال سعودي. مدة التوريد 30 يوماً من تاريخ توقيع العقد.
"""

prompt = PROMPT_TEMPLATE.format(
    schema=JSON_SCHEMA_DESCRIPTION,
    text=new_contract_text,
    output="",
)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(result)


## 10. Diagnostic visualization — train vs. validation loss

If validation loss flattens or rises while training loss keeps falling, the
model is overfitting and epochs/LoRA rank/learning rate need adjusting.


In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_steps = [h["step"] for h in history if "loss" in h]
train_loss  = [h["loss"] for h in history if "loss" in h]

eval_steps  = [h["step"] for h in history if "eval_loss" in h]
eval_loss   = [h["eval_loss"] for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 5))
plt.plot(train_steps, train_loss, label="Training Loss", color="tab:blue")
plt.plot(eval_steps, eval_loss, label="Validation Loss", color="tab:orange", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
